# 01 · Bronze Ingestion – Hire-to-Retire (H2R)
**Layer:** Bronze (raw, immutable) · **Engine:** Fabric Spark · **Lakehouse:** `lh_h2r`

Loads every raw CSV landed by the Data Factory pipeline in `Files/raw/h2r/` into Delta tables, **as-is**,
adding only audit columns. No business rules are applied here – Bronze is the replayable source of truth.

| Source system | Files |
|---|---|
| HRIS | employees, departments, locations, grades, onboarding, exits |
| Recruitment (ATS) | requisitions, candidates |
| Learning (LMS) | courses, training_records |
| Time & Attendance | leave_records |
| Performance | reviews |
| Payroll | payroll_monthly |

In [ ]:
# Parameters (overridden by the Data Factory pipeline at run time)
raw_path   = "Files/raw/h2r"
batch_id   = "manual_run"
load_mode  = "overwrite"     # overwrite = full load, append = incremental

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime

SOURCES = {
    "hris_employees":       "HRIS",
    "hris_departments":     "HRIS",
    "hris_locations":       "HRIS",
    "hris_grades":          "HRIS",
    "hris_onboarding":      "HRIS",
    "hris_exits":           "HRIS",
    "ats_requisitions":     "ATS",
    "ats_candidates":       "ATS",
    "lms_courses":          "LMS",
    "lms_training_records": "LMS",
    "tna_leave_records":    "TNA",
    "perf_reviews":         "PERFORMANCE",
    "payroll_monthly":      "PAYROLL",
}
run_ts = datetime.utcnow()

## Ingest each source file into a `bronze_*` Delta table

In [ ]:
audit = []
for src, system in SOURCES.items():
    path = f"{raw_path}/{src}.csv"
    df = (spark.read
            .option("header", True)
            .option("inferSchema", False)        # keep everything as string in Bronze
            .option("mode", "PERMISSIVE")
            .csv(path))

    # Spark/Delta-friendly column names
    for c in df.columns:
        df = df.withColumnRenamed(c, c.strip().lower().replace(" ", "_"))

    df = (df.withColumn("_source_system", F.lit(system))
            .withColumn("_source_file",   F.lit(path))
            .withColumn("_batch_id",      F.lit(batch_id))
            .withColumn("_ingested_at",   F.lit(run_ts).cast("timestamp")))

    target = f"bronze_{src}"
    (df.write.format("delta")
        .mode(load_mode)
        .option("overwriteSchema", "true")
        .saveAsTable(target))

    rows = spark.table(target).count()
    audit.append((batch_id, target, system, rows, run_ts))
    print(f"✔ {target:<32} {rows:>8,} rows")

## Write ingestion audit log

In [ ]:
audit_df = spark.createDataFrame(audit, "batch_id string, table_name string, source_system string, row_count long, loaded_at timestamp")
audit_df.write.format("delta").mode("append").saveAsTable("ops_ingestion_audit")
display(audit_df)